In [8]:
import os, getpass, random, re, json
from dotenv import load_dotenv
import re, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

# Get LLM

In [4]:
load_dotenv()  # reads LLM_PROVIDER from .env (no keys stored there)

PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
if PROVIDER not in ("groq", "openai"):
    raise ValueError(f"LLM_PROVIDER must be 'groq' or 'openai', got '{PROVIDER}'")

# Key comes from the environment if already set this session, otherwise a hidden prompt.
KEY_NAME = "GROQ_API_KEY" if PROVIDER == "groq" else "OPENAI_API_KEY"
if not os.environ.get(KEY_NAME):
    os.environ[KEY_NAME] = getpass.getpass(f"Enter your {KEY_NAME}: ")

if PROVIDER == "groq":
    from groq import Groq
    client = Groq()          # reads GROQ_API_KEY
    MODEL = "openai/gpt-oss-120b"
else:
    from openai import OpenAI
    client = OpenAI()        # reads OPENAI_API_KEY
    MODEL = "YOUR-OPENAI-MODEL"  # fill in a model from platform.openai.com/docs/models

def chat(messages, tools=None, tool_choice="auto", temperature=0.0, max_tokens=1024):
    """Thin wrapper around one chat-completion call. Returns the message object."""
    kwargs = dict(model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens)
    if "gpt-oss" in MODEL:
        kwargs["reasoning_effort"] = "low"
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs).choices[0].message

rng = random.Random(7)      # fixed seed

# Sanity check (a real API call):
print(f"Provider: {PROVIDER}, model: {MODEL}")
print(chat([{"role": "user", "content": "Reply with exactly one word: ready"}]).content)

Provider: groq, model: openai/gpt-oss-120b
ready


# Vanilla RAG 1

## Chunking

In [7]:
text = open("data/nvda_2026/full_text.txt", encoding="utf-8").read()

def table_to_text(i):
    df = pd.read_csv(f"data/nvda_2026/tables/table_{i}.csv", header=None)
    df = df.dropna(how="all").dropna(axis=1, how="all")
    rows = [" | ".join(str(c) for c in r if pd.notna(c) and str(c).strip() not in ("$", ")", "%"))
            for r in df.values]
    return " ; ".join(rows)

# put each table back into the text where its [TABLE_n] marker was
text = re.sub(r"\[TABLE_(\d+)\]", 
              lambda m: " " + table_to_text(int(m.group(1))) + " ", 
              text)

def chunk_words(text, size=150, overlap=30):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), step)]

chunks = chunk_words(text)
print(len(chunks), "chunks")
print(chunks[100][:500])

612 chunks
and regional basis. Extended lead times may occur if we experience other supply constraints caused by natural disasters, pandemics or other events. Geopolitical tensions in regions where we rely on suppliers, contract manufacturers, and assembly partners that are critical to our supply continuity, could have a material adverse impact on us. Publicly announced intentions by governments or other companies to purchase our products can further complicate our demand estimates, as such announcements a


## Chunks to embeddings

In [9]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vecs = embedder.encode(chunks, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print(chunk_vecs.shape)     # (number of chunks, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

(612, 384)


## Test if embeddings / cosine simliarity is able to retrieve relevant answer

In [10]:
def retrieve(query, k=5):
    q = embedder.encode([query], normalize_embeddings=True)[0]    # same model as the chunks
    scores = chunk_vecs @ q
    idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), chunks[i]) for i in idx]

for i, s, c in retrieve("What are NVIDIA's main risk factors related to export controls?"):
    print(i, round(s, 3), c[:150], "\n")

189 0.751 export controls may have a greater impact on our ability to compete in markets subject to those controls. Export controls may disrupt our supply and d 

173 0.643 and/or ability to provide services such as NVIDIA AI cloud services and could impact the cost and/or ability for our CSPs and customers to provide ser 

172 0.631 that limit our ability to serve demand abroad and could negatively and materially impact our business, revenue and financial results. Export controls  

188 0.623 in popularity with developers worldwide. Any regulatory control or other restriction that limits our ability to provide products and services that sup 

190 0.614 certain of our competitors that sell chips that are outside the scope of such control. Export controls have already and may in the future encourage cu 



In [22]:
RAG_SYSTEM = ("Answer the question using ONLY the context below. Cite the chunk ids you used, like [12]. "
              "If the answer is not in the context, reply exactly: Not found in the provided context.")

def rag_answer(question, k=5):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{i}] {c}" for i, s, c in hits)
    msg = chat([{"role": "system", "content": RAG_SYSTEM},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}],
               max_tokens=1500)
    return {"answer": msg.content, "hits": hits}

out = rag_answer("key strategies that shape nvidia's overall business approach")
# out = rag_answer("What was NVIDIA's total revenue for the year ending 25 Jan 2026?")
# out = rag_answer("What is NVIDIA's key business strategies?")
print(out["answer"])
# print(out["hits"])
for hit in out["hits"]:
    print(f"\n{hit}")

NVIDIA’s overall business approach is built around a few core strategies that are repeatedly highlighted in the filing:

* **Leverage its GPU architecture to expand beyond PC graphics** – The company uses the same accelerated‑computing platform that powers high‑end 3‑D graphics to create solutions for scientific computing, artificial‑intelligence, data science, autonomous vehicles, robotics and digital‑twin applications, positioning itself as a “data‑center‑scale AI infrastructure” company [240].

* **Focus on two primary operating segments** – All of its products and services are organized under the “Compute & Networking” segment and the “Graphics” segment, allowing NVIDIA to target both AI‑driven data‑center workloads and traditional graphics markets [240].

* **Build a broad ecosystem of direct and indirect customers** – NVIDIA sells directly to AIBs, distributors, ODMs, OEMs, cloud service providers, AI model makers and system integrators, while also serving indirect customers (clo

In [21]:
phrase = "key strategies"
found = [i for i, c in enumerate(chunks) if phrase in c.lower()]
print("chunks containing the phrase:", found)

question = "What are NVIDIA's key business strategies?"
q = embedder.encode([question], normalize_embeddings=True)[0]
order = np.argsort(-(chunk_vecs @ q))

for i in found:
    print("chunk", i, "rank", int(np.where(order == i)[0][0]) + 1, "of", len(chunks))
    print(chunks[i][:300], "\n")

chunks containing the phrase: [39]
chunk 39 rank 50 of 612
the same Tensor Core technology found in our Data Center solutions. Automotive Automotive is comprised of platform solutions for automated driving from the cloud to the car. Leveraging our technology leadership in AI and building on long-standing relationships across several hundred automotive ecosy 



# Vanilla RAG 2

Change the transformer model from `all-MiniLM-L6-v2` to `BAAI/bge-small-en-v1.5` and add QUERY_PREFIX because of how this transformer model was trained. 

In [23]:
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
chunk_vecs = embedder.encode(chunks, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print(chunk_vecs.shape)     # (number of chunks, 384)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

(612, 384)


In [24]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def retrieve(query, k=5):
    q = embedder.encode([QUERY_PREFIX + query], normalize_embeddings=True)[0] 
    scores = chunk_vecs @ q
    idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), chunks[i]) for i in idx]

for i, s, c in retrieve("What are NVIDIA's main risk factors related to export controls?"):
    print(i, round(s, 3), c[:150], "\n")

172 0.822 that limit our ability to serve demand abroad and could negatively and materially impact our business, revenue and financial results. Export controls  

190 0.798 certain of our competitors that sell chips that are outside the scope of such control. Export controls have already and may in the future encourage cu 

189 0.794 export controls may have a greater impact on our ability to compete in markets subject to those controls. Export controls may disrupt our supply and d 

173 0.779 and/or ability to provide services such as NVIDIA AI cloud services and could impact the cost and/or ability for our CSPs and customers to provide ser 

188 0.777 in popularity with developers worldwide. Any regulatory control or other restriction that limits our ability to provide products and services that sup 



In [25]:
RAG_SYSTEM = ("Answer the question using ONLY the context below. Cite the chunk ids you used, like [12]. "
              "If the answer is not in the context, reply exactly: Not found in the provided context.")

def rag_answer(question, k=5):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{i}] {c}" for i, s, c in hits)
    msg = chat([{"role": "system", "content": RAG_SYSTEM},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}],
               max_tokens=1500)
    return {"answer": msg.content, "hits": hits}

out = rag_answer("key strategies that shape nvidia's overall business approach")
# out = rag_answer("What was NVIDIA's total revenue for the year ending 25 Jan 2026?")
# out = rag_answer("What is NVIDIA's key business strategies?")
print(out["answer"])
# print(out["hits"])
for hit in out["hits"]:
    print(f"\n{hit}")

NVIDIA’s overall business approach is driven by a few core strategies:

* **Advancing the NVIDIA accelerated‑computing platform** – delivering performance leaps that outpace Moore’s Law by innovating across the entire stack (architecture, chip design, system, interconnect, algorithms and software) to give order‑of‑magnitude advantages in its target markets【40】.  
* **A full‑stack, unified programmable architecture** – using the same underlying GPUs, CPUs, CUDA and networking technologies for diverse markets (data‑center, gaming, professional visualization, automotive, robotics, etc.) so that investments can be leveraged across multiple multi‑billion‑dollar segments【40】.  
* **A platform strategy that brings together hardware, systems, software, algorithms, libraries, AI models, training data sets and services** – creating unique value for customers and enabling AI solutions such as generative and agentic AI across many industries【22】.  

These strategies together shape NVIDIA’s busines

# Vanilla RAG 3

Change the chunking from a fixed word count to chunking pegged to document structure so that parts do not get cut off

In [26]:
raw = open("data/nvda_2026/full_text.txt", encoding="utf-8").read()

def table_to_text(i):
    df = pd.read_csv(f"data/nvda_2026/tables/table_{i}.csv", header=None)
    df = df.dropna(how="all").dropna(axis=1, how="all")
    rows = [" | ".join(str(c) for c in r if pd.notna(c) and str(c).strip() not in ("$", ")", "%"))
            for r in df.values]
    return " ; ".join(rows)

def is_heading(line):
    # short line that doesn't end like a sentence, e.g. "Business Strategies"
    words = line.split()
    return (1 <= len(words) <= 10
            and not line.endswith((".", ":", ";", ","))
            and not re.fullmatch(r"[\d\W]+", line))

def chunk_by_structure(raw, max_words=200):
    chunks, buf = [], []
    item, heading = "", ""

    def label():
        return " > ".join(p for p in (item, heading) if p)

    def flush():
        if buf:
            chunks.append({"section": label(), "text": f"{label()}\n" + " ".join(buf)})
            buf.clear()

    for line in raw.splitlines():
        line = line.strip()
        if not line or line == "Table of Contents" or line.isdigit():
            continue                                             # page furniture
        if re.match(r"^Item \d+[A-C]?\.", line):                 # 10-K part, e.g. "Item 1A. Risk Factors"
            flush(); item, heading = line, ""
        elif m := re.fullmatch(r"\[TABLE_(\d+)\]", line):          # each table becomes its own chunk
            flush()
            chunks.append({"section": label(), "text": f"{label()} (table)\n" + table_to_text(int(m.group(1)))})
        elif is_heading(line):
            flush(); heading = line
        else:
            if buf and len(" ".join(buf).split()) + len(line.split()) > max_words:
                flush()
            buf.append(line)
    flush()
    return chunks

docs = chunk_by_structure(raw)
chunks = [d["text"] for d in docs]       # same name as before, so embed/retrieve cells work unchanged
print(len(chunks), "chunks")
print(chunks[41][:300])


508 chunks
Item 1. Business > Business Strategies
NVIDIA’s key strategies that shape our overall business approach include: Advancing the NVIDIA accelerated computing platform. Our accelerated computing platform can solve complex problems in significantly less time and with lower power consumption than alterna


In [27]:
for i, s, c in retrieve("What are NVIDIA's main risk factors related to export controls?"):
    print(i, round(s, 3), c[:150], "\n")

172 0.822 Item 1A. Risk Factors > Risks Related to Regulatory, Legal, Our Stock and Other Matters
We are subject to complex income tax laws and regulations, as  

190 0.798 Item 5. Market for Registrant's Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities > Recent Sales of Unregistered Se 

189 0.794 Item 5. Market for Registrant's Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities > Restricted Stock Unit Share Wit 

173 0.779 Item 1A. Risk Factors > Risks Related to Regulatory, Legal, Our Stock and Other Matters
Further, changes in tax laws or their interpretation by tax au 

188 0.777 Item 5. Market for Registrant's Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities > Issuer Purchases of Equity Secu 



In [28]:
RAG_SYSTEM = ("Answer the question using ONLY the context below. Cite the chunk ids you used, like [12]. "
              "If the answer is not in the context, reply exactly: Not found in the provided context.")

def rag_answer(question, k=5):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{i}] {c}" for i, s, c in hits)
    msg = chat([{"role": "system", "content": RAG_SYSTEM},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}],
               max_tokens=1500)
    return {"answer": msg.content, "hits": hits}

out = rag_answer("key strategies that shape nvidia's overall business approach")
# out = rag_answer("What was NVIDIA's total revenue for the year ending 25 Jan 2026?")
# out = rag_answer("What is NVIDIA's key business strategies?")
print(out["answer"])
# print(out["hits"])
for hit in out["hits"]:
    print(f"\n{hit}")

NVIDIA’s overall business approach is built around a few core strategic pillars that appear throughout the filing:

* **Leverage AI leadership to create end‑to‑end platform solutions** – NVIDIA is using its long‑standing AI expertise and a broad ecosystem of automotive partners to deliver a full‑stack autonomous‑driving offering (the DRIVE Hyperion platform). This includes development infrastructure, high‑performance, energy‑efficient DRIVE AGX hardware, an in‑vehicle operating system, a reference sensor suite and an open, modular software stack for self‑driving, mapping, parking and intelligent in‑vehicle experiences【40】.  

* **Expand liquidity and financing flexibility** – The company has increased its unsecured commercial‑paper program to as much as $25 billion, giving it a flexible source of short‑term funding that it can draw on as part of its overall liquidity strategy【240】.  

* **Manage supply‑chain risk and demand forecasting** – NVIDIA acknowledges that mismatches between de